# MD and harmonic heat-capacity analysis

This notebook discovers completed calculations below `output/` and compares all available MD logs and harmonic heat-capacity (`.npz`) results. It does not assume a particular run name or a fixed set of temperatures.

The comparative analysis reads only the small log and result files. The final, optional section samples one trajectory for force and Chemiscope inspection, avoiding loading every large trajectory into memory.

In [ ]:
import json
from pathlib import Path
import re

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

# User settings
OUTPUT_DIR = None  # None -> locate the project's output/ directory automatically
EQUILIBRATION_FRACTION = 0.5  # discard this initial fraction in MD summaries
SELECTED_RUN = None  # e.g. 'mof5-100ch4-300K-test'; None prefers the run nearest 300 K
TRAJECTORY_STRIDE = 100  # sample every Nth frame for force/viewer analysis
ZERO_FREQUENCY_TOLERANCE_CM1 = 1e-6


def find_project_dir(start=Path.cwd()):
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'output').is_dir() and (candidate / 'run.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not locate a project directory containing run.py and output/. '
        'Set OUTPUT_DIR explicitly in this cell.'
    )


PROJECT_DIR = find_project_dir() if OUTPUT_DIR is None else Path(OUTPUT_DIR).expanduser().resolve().parent
OUTPUT_DIR = PROJECT_DIR / 'output' if OUTPUT_DIR is None else Path(OUTPUT_DIR).expanduser().resolve()
if not 0.0 <= EQUILIBRATION_FRACTION < 1.0:
    raise ValueError('EQUILIBRATION_FRACTION must be in [0, 1)')
if TRAJECTORY_STRIDE < 1:
    raise ValueError('TRAJECTORY_STRIDE must be positive')
print(f'Project: {PROJECT_DIR}')
print(f'Outputs: {OUTPUT_DIR}')

## Discover runs and load results

A run folder is any directory below `output/` containing an MD log, ASE trajectory, or heat-capacity archive. Target MD temperatures are inferred from names containing a token such as `300K`; runs without such a token are still included.

In [ ]:
TEMPERATURE_PATTERN = re.compile(r'(?<![0-9.])(\d+(?:\.\d+)?)\s*K(?:\b|_)', re.IGNORECASE)


def infer_temperature(*values):
    for value in values:
        match = TEMPERATURE_PATTERN.search(str(value))
        if match:
            return float(match.group(1))
    return np.nan


def read_md_log(path):
    data = np.atleast_2d(np.loadtxt(path, skiprows=1, dtype=float))
    if data.shape[1] != 5:
        raise ValueError(f'{path} has {data.shape[1]} columns; expected 5')
    if len(data) == 0 or not np.all(np.isfinite(data)):
        raise ValueError(f'{path} is empty or contains non-finite values')
    return {
        'time_ps': data[:, 0],
        'total_energy_eV': data[:, 1],
        'potential_energy_eV': data[:, 2],
        'kinetic_energy_eV': data[:, 3],
        'temperature_K': data[:, 4],
    }


def scalar_text(value):
    return str(np.asarray(value).reshape(-1)[0])


def read_heat_capacity(path):
    required = {'temperatures_K', 'cv_J_per_gK', 'frame_indices', 'frequencies_cm1'}
    with np.load(path, allow_pickle=False) as archive:
        missing = required.difference(archive.files)
        if missing:
            raise ValueError(f'{path} is missing arrays: {sorted(missing)}')
        temperatures = np.asarray(archive['temperatures_K'], dtype=float).reshape(-1)
        cv = np.atleast_2d(np.asarray(archive['cv_J_per_gK'], dtype=float))
        frames = np.asarray(archive['frame_indices'], dtype=int).reshape(-1)
        frequencies = np.atleast_2d(np.asarray(archive['frequencies_cm1'], dtype=float))
        run_name = scalar_text(archive['run_name']) if 'run_name' in archive else path.parent.name
        metadata = json.loads(scalar_text(archive['metadata'])) if 'metadata' in archive else {}
    if cv.shape != (len(frames), len(temperatures)):
        raise ValueError(f'{path}: C_V shape {cv.shape} does not match frames and temperatures')
    if frequencies.shape[0] != len(frames):
        raise ValueError(f'{path}: frequency rows do not match frame indices')
    if np.any(np.diff(temperatures) <= 0) or not np.all(np.isfinite(cv)):
        raise ValueError(f'{path}: temperatures must increase and C_V values must be finite')
    return {
        'path': path, 'run_name': run_name, 'temperatures_K': temperatures,
        'cv_J_per_gK': cv, 'frame_indices': frames,
        'frequencies_cm1': frequencies, 'metadata': metadata,
    }

In [ ]:
candidate_files = [
    path for pattern in ('*.log', '*.traj', '*.npz')
    for path in OUTPUT_DIR.rglob(pattern)
]
run_directories = sorted({path.parent for path in candidate_files})
runs = []
for directory in run_directories:
    logs = sorted(directory.glob('*.log'))
    trajectories = sorted(directory.glob('*.traj'))
    heat_paths = sorted(directory.glob('*.npz'))
    if not (logs or trajectories or heat_paths):
        continue
    run = {
        'name': directory.name,
        'directory': directory,
        'target_temperature_K': infer_temperature(directory.name),
        'log_paths': logs,
        'trajectory_paths': trajectories,
        'heat_capacity_paths': heat_paths,
        'md': [],
        'heat_capacity': [],
        'errors': [],
    }
    for path in logs:
        try:
            run['md'].append({'path': path, **read_md_log(path)})
        except Exception as error:
            run['errors'].append(f'{path.name}: {error}')
    for path in heat_paths:
        try:
            result = read_heat_capacity(path)
            if np.isnan(run['target_temperature_K']):
                run['target_temperature_K'] = infer_temperature(result['run_name'], path.name)
            run['heat_capacity'].append(result)
        except Exception as error:
            run['errors'].append(f'{path.name}: {error}')
    runs.append(run)

if not runs:
    raise FileNotFoundError(f'No .log, .traj, or .npz files found below {OUTPUT_DIR}')
runs.sort(key=lambda run: (np.isnan(run['target_temperature_K']), run['target_temperature_K'], run['name']))

lines = [
    '| run | target T [K] | logs | trajectories | heat-capacity files | status |',
    '|---|---:|---:|---:|---:|---|',
]
for run in runs:
    target = '—' if np.isnan(run['target_temperature_K']) else f"{run['target_temperature_K']:g}"
    status = '<br>'.join(run['errors']) if run['errors'] else 'OK'
    lines.append(
        f"| `{run['name']}` | {target} | {len(run['md'])}/{len(run['log_paths'])} | "
        f"{len(run['trajectory_paths'])} | {len(run['heat_capacity'])}/{len(run['heat_capacity_paths'])} | {status} |"
    )
display(Markdown('\n'.join(lines)))

## Molecular-dynamics diagnostics

The plots compare temperature and energy changes across all readable logs. Summary statistics use only the final `1 - EQUILIBRATION_FRACTION` portion. Energy drift is the slope of a least-squares line over that same interval; it is a diagnostic, not an NVT energy-conservation test.

In [ ]:
md_records = []
for run in runs:
    for md in run['md']:
        start = min(int(len(md['time_ps']) * EQUILIBRATION_FRACTION), len(md['time_ps']) - 1)
        selection = slice(start, None)
        time = md['time_ps'][selection]
        energy = md['total_energy_eV'][selection]
        drift = np.polyfit(time, energy, 1)[0] if len(np.unique(time)) > 1 else np.nan
        md_records.append({
            'run': run, 'md': md,
            'mean_temperature_K': np.mean(md['temperature_K'][selection]),
            'std_temperature_K': np.std(md['temperature_K'][selection], ddof=1) if len(time) > 1 else 0.0,
            'energy_drift_eV_per_ps': drift,
            'duration_ps': md['time_ps'][-1] - md['time_ps'][0],
            'records': len(md['time_ps']),
        })

if not md_records:
    print('No readable MD logs were found; skipping MD plots.')
else:
    colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(md_records)))
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
    for record, color in zip(md_records, colors, strict=True):
        run, md = record['run'], record['md']
        target = run['target_temperature_K']
        label = run['name']
        axes[0, 0].plot(md['time_ps'], md['temperature_K'], color=color, alpha=0.85, label=label)
        axes[0, 1].plot(
            md['time_ps'], md['total_energy_eV'] - md['total_energy_eV'][0],
            color=color, alpha=0.85, label=label,
        )
        axes[1, 0].plot(
            md['time_ps'], md['potential_energy_eV'] - md['potential_energy_eV'][0],
            color=color, alpha=0.85, label=label,
        )
        if np.isfinite(target):
            axes[1, 1].errorbar(
                target, record['mean_temperature_K'], yerr=record['std_temperature_K'],
                fmt='o', capsize=3, color=color,
            )
    finite_targets = np.array([r['run']['target_temperature_K'] for r in md_records], dtype=float)
    finite_targets = finite_targets[np.isfinite(finite_targets)]
    if len(finite_targets):
        limits = [finite_targets.min(), finite_targets.max()]
        if limits[0] == limits[1]:
            limits = [limits[0] - 1, limits[1] + 1]
        axes[1, 1].plot(limits, limits, '--', color='0.35', label='target')
    axes[0, 0].set(xlabel='Time [ps]', ylabel='Temperature [K]', title='Temperature traces')
    axes[0, 1].set(xlabel='Time [ps]', ylabel=r'$E_{tot}(t)-E_{tot}(0)$ [eV]', title='Total-energy change')
    axes[1, 0].set(xlabel='Time [ps]', ylabel=r'$E_{pot}(t)-E_{pot}(0)$ [eV]', title='Potential-energy change')
    axes[1, 1].set(xlabel='Target MD temperature [K]', ylabel='Production mean temperature [K]', title='Thermostat tracking')
    axes[0, 0].legend(fontsize='small', ncols=2)
    axes[1, 1].legend(fontsize='small')
    plt.show()

    lines = [
        '| run | records | duration [ps] | production T [K] | T std [K] | E drift [eV/ps] |',
        '|---|---:|---:|---:|---:|---:|',
    ]
    for record in md_records:
        lines.append(
            f"| `{record['run']['name']}` | {record['records']} | {record['duration_ps']:.4g} | "
            f"{record['mean_temperature_K']:.2f} | {record['std_temperature_K']:.2f} | "
            f"{record['energy_drift_eV_per_ps']:.4g} |"
        )
    display(Markdown('\n'.join(lines)))

## Harmonic heat capacity

Each thin line is one analyzed trajectory frame; the thicker line is the mean when an archive contains multiple frames. The right panel evaluates each curve at the MD sampling temperature, when that temperature lies within the calculated grid. Results from different MD temperatures describe different instantaneous structures and should only be combined after structural stability and frame convergence have been established.

In [ ]:
heat_records = [
    {'run': run, 'result': result}
    for run in runs for result in run['heat_capacity']
]
matched_cv = []
if not heat_records:
    print('No readable heat-capacity archives were found; skipping C_V plots.')
else:
    run_names = list(dict.fromkeys(record['run']['name'] for record in heat_records))
    color_map = {name: color for name, color in zip(
        run_names, plt.cm.plasma(np.linspace(0.08, 0.9, len(run_names))), strict=True
    )}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
    for record in heat_records:
        run, result = record['run'], record['result']
        temperature = result['temperatures_K']
        cv = result['cv_J_per_gK']
        color = color_map[run['name']]
        for curve in cv:
            axes[0].plot(temperature, curve, color=color, alpha=0.22, linewidth=1)
        axes[0].plot(temperature, cv.mean(axis=0), color=color, linewidth=2, label=run['name'])
        target = run['target_temperature_K']
        if np.isfinite(target) and temperature[0] <= target <= temperature[-1]:
            values = np.array([np.interp(target, temperature, curve) for curve in cv])
            matched_cv.append({
                'run': run, 'path': result['path'], 'frames': result['frame_indices'],
                'temperature_K': target, 'values': values,
            })
            axes[1].errorbar(
                target, values.mean(), yerr=values.std(ddof=1) if len(values) > 1 else None,
                fmt='o', capsize=3, color=color,
            )
    axes[0].set(xlabel='Analysis temperature [K]', ylabel=r'$C_V$ [J g$^{-1}$ K$^{-1}$]', title='Harmonic heat-capacity curves')
    axes[1].set(xlabel='MD sampling temperature [K]', ylabel=r'$C_V(T_{MD})$ [J g$^{-1}$ K$^{-1}$]', title=r'$C_V$ at each sampling temperature')
    axes[0].legend(fontsize='small')
    plt.show()

    lines = [
        '| run | archive | frames | sampling T [K] | mean C_V(T_MD) [J/gK] | frame std |',
        '|---|---|---|---:|---:|---:|',
    ]
    for record in matched_cv:
        values = record['values']
        frame_text = ', '.join(map(str, record['frames']))
        std = np.std(values, ddof=1) if len(values) > 1 else np.nan
        lines.append(
            f"| `{record['run']['name']}` | `{record['path'].name}` | {frame_text} | "
            f"{record['temperature_K']:g} | {values.mean():.6f} | "
            f"{'—' if np.isnan(std) else f'{std:.6f}'} |"
        )
    if matched_cv:
        display(Markdown('\n'.join(lines)))
    else:
        print('No inferred MD temperature fell inside its heat-capacity temperature grid.')

## Vibrational-frequency diagnostics

The histogram excludes near-zero modes so that the nonzero spectrum remains visible. The table reports non-finite and near-zero values explicitly. A large zero-mode count, unexpected imaginary-mode convention, or strong spectrum change between frames should be investigated before interpreting the harmonic heat capacity.

In [ ]:
if heat_records:
    all_finite = np.concatenate([
        record['result']['frequencies_cm1'][np.isfinite(record['result']['frequencies_cm1'])]
        for record in heat_records
    ])
    nonzero = all_finite[np.abs(all_finite) > ZERO_FREQUENCY_TOLERANCE_CM1]
    if len(nonzero):
        lower, upper = np.percentile(nonzero, [0.5, 99.5])
        bins = np.linspace(lower, upper, 80)
        fig, ax = plt.subplots(figsize=(10, 4.5), layout='constrained')
        for record in heat_records:
            frequencies = record['result']['frequencies_cm1'].reshape(-1)
            frequencies = frequencies[
                np.isfinite(frequencies) & (np.abs(frequencies) > ZERO_FREQUENCY_TOLERANCE_CM1)
            ]
            ax.hist(
                frequencies, bins=bins, density=True, histtype='step', linewidth=1.5,
                color=color_map[record['run']['name']], label=record['run']['name'],
            )
        ax.set(xlabel=r'Frequency [cm$^{-1}$]', ylabel='Probability density', title='Nonzero vibrational-frequency distributions')
        ax.legend(fontsize='small')
        plt.show()

    lines = [
        '| run | archive | frames | modes/frame | near zero | negative | non-finite | range [cm⁻¹] |',
        '|---|---|---:|---:|---:|---:|---:|---:|',
    ]
    for record in heat_records:
        result = record['result']
        frequencies = result['frequencies_cm1']
        finite = frequencies[np.isfinite(frequencies)]
        near_zero = np.count_nonzero(np.isfinite(frequencies) & (np.abs(frequencies) <= ZERO_FREQUENCY_TOLERANCE_CM1))
        negative = np.count_nonzero(frequencies < -ZERO_FREQUENCY_TOLERANCE_CM1)
        nonfinite = frequencies.size - finite.size
        value_range = '—' if not len(finite) else f'{finite.min():.2f} to {finite.max():.2f}'
        lines.append(
            f"| `{record['run']['name']}` | `{result['path'].name}` | {len(result['frame_indices'])} | "
            f"{frequencies.shape[1]} | {near_zero} | {negative} | {nonfinite} | {value_range} |"
        )
    display(Markdown('\n'.join(lines)))

## Sample one trajectory

Set `SELECTED_RUN` and `TRAJECTORY_STRIDE` in the first cell if needed. Only sampled frames are read. Force availability depends on whether the calculator results were stored in the ASE trajectory.

In [ ]:
trajectory_runs = [run for run in runs if run['trajectory_paths']]
if not trajectory_runs:
    selected_run = None
    viewer_frames = []
    print('No ASE trajectories were found; skipping trajectory analysis.')
else:
    if SELECTED_RUN is not None:
        matches = [run for run in trajectory_runs if run['name'] == SELECTED_RUN]
        if not matches:
            raise ValueError(f'SELECTED_RUN={SELECTED_RUN!r} does not identify a run with a trajectory')
        selected_run = matches[0]
    else:
        selected_run = min(
            trajectory_runs,
            key=lambda run: abs(run['target_temperature_K'] - 300.0)
            if np.isfinite(run['target_temperature_K']) else np.inf,
        )
    trajectory_path = selected_run['trajectory_paths'][0]
    import ase.io

    viewer_frames = ase.io.read(trajectory_path, index=f'::{TRAJECTORY_STRIDE}')
    if not viewer_frames:
        raise ValueError(f'No frames found in {trajectory_path}')
    force_frames = []
    maximum_forces = []
    for frame in viewer_frames:
        try:
            forces = frame.get_forces()
        except Exception:
            force_frames = []
            maximum_forces = []
            break
        force_frames.append(forces)
        maximum_forces.append(np.linalg.norm(forces, axis=1).max())

    print(f'Selected trajectory: {trajectory_path.relative_to(PROJECT_DIR)}')
    print(f'Sampled frames: {len(viewer_frames)} (stride {TRAJECTORY_STRIDE})')
    print(f'Atoms per frame: {len(viewer_frames[0])}')
    print(f'Cell volume: {viewer_frames[0].cell.volume:.3f} Å³')
    print(f"Elements: {sorted(set(viewer_frames[0].get_chemical_symbols()))}")
    if maximum_forces:
        selected_md = selected_run['md'][0] if selected_run['md'] else None
        if selected_md is not None:
            sample_indices = np.arange(len(viewer_frames)) * TRAJECTORY_STRIDE
            sample_indices = np.minimum(sample_indices, len(selected_md['time_ps']) - 1)
            force_time = selected_md['time_ps'][sample_indices]
        else:
            force_time = np.arange(len(viewer_frames))
        plt.figure(figsize=(10, 3.5), layout='constrained')
        plt.plot(force_time, maximum_forces, marker='o', markersize=3)
        plt.xlabel('Time [ps]' if selected_md is not None else 'Sampled frame')
        plt.ylabel('Maximum force [eV/Å]')
        plt.title(f"Maximum atomic force — {selected_run['name']}")
        plt.show()
    else:
        print('Forces are not stored in the sampled trajectory.')

### Optional Chemiscope viewer and export

This cell creates an interactive viewer for the sampled frames and saves a portable `.json.gz` dataset inside the selected run folder.

In [ ]:
if viewer_frames:
    try:
        import chemiscope
    except ImportError:
        print('Chemiscope is not installed; install it to enable the interactive viewer.')
    else:
        frame_count = len(viewer_frames)
        properties = {'sampled frame': np.arange(frame_count)}
        selected_md = selected_run['md'][0] if selected_run['md'] else None
        if selected_md is not None:
            sample_indices = np.minimum(
                np.arange(frame_count) * TRAJECTORY_STRIDE, len(selected_md['time_ps']) - 1
            )
            properties.update({
                'time': {'target': 'structure', 'values': selected_md['time_ps'][sample_indices], 'units': 'ps'},
                'potential energy': {'target': 'structure', 'values': selected_md['potential_energy_eV'][sample_indices], 'units': 'eV'},
                'temperature': {'target': 'structure', 'values': selected_md['temperature_K'][sample_indices], 'units': 'K'},
            })
        shapes = {}
        if force_frames and len(force_frames) == frame_count:
            for frame, forces in zip(viewer_frames, force_frames, strict=True):
                frame.arrays['forces_eV_per_A'] = forces
            shapes['forces'] = chemiscope.ase_vectors_to_arrows(
                viewer_frames, 'forces_eV_per_A', scale=1.0, radius=0.08
            )
        structure_settings = {'keepOrientation': True, 'playbackDelay': 150}
        if shapes:
            structure_settings['shape'] = 'forces'
        viewer = chemiscope.show(
            structures=viewer_frames, properties=properties, shapes=shapes,
            settings={'structure': [structure_settings], 'map': {'joinPoints': True}},
            mode='default',
        )
        display(viewer)
        chemiscope_path = selected_run['directory'] / f"{trajectory_path.stem}-sampled.json.gz"
        viewer.save(str(chemiscope_path))
        print(f'Saved Chemiscope dataset: {chemiscope_path.relative_to(PROJECT_DIR)}')

## Interpretation checklist

Before treating these values as converged material properties, check that:

- the post-equilibration temperatures fluctuate around their targets without systematic drift;
- structures and maximum forces remain physically reasonable;
- heat capacities are stable across several decorrelated frames, not just one final frame;
- the vibrational spectrum and zero/imaginary-mode treatment are understood; and
- MD duration, system size, model, Hessian precision, and sparse-Hessian settings have been convergence-tested.

The current repository configurations are short integration tests, so their plots validate the workflow but do not by themselves establish production-quality convergence.